<a href="https://colab.research.google.com/github/gkkg19/Deep-Learning/blob/main/L2_Regularization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import time

# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train={X_train.shape[0]}, Test={X_test.shape[0]}, Features={X_train.shape[1]}")

# Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

# Train single-layer perceptron with L2 regularization
def train_perceptron_l2(X_train, X_test, y_train, y_test, lambda_l2=0.0, lr=0.5, epochs=1000):
    np.random.seed(42)
    n_features = X_train.shape[1]

    # Initialize weights and bias
    W = np.random.randn(n_features, 1) * 0.1
    b = 0.0
    m = X_train.shape[0]

    start_time = time.time()
    for epoch in range(epochs):
        # Forward
        O = sigmoid(X_train @ W + b)

        # Backward
        dO = (O - y_train.reshape(-1,1)) / m * O * (1 - O)
        dW = X_train.T @ dO
        db = np.sum(dO)

        # L2 regularization
        dW += lambda_l2 * W

        # Update
        W -= lr * dW
        b -= lr * db

    train_time = time.time() - start_time

    # Compute accuracy
    train_pred = np.round(sigmoid(X_train @ W + b))
    test_pred = np.round(sigmoid(X_test @ W + b))
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    feature_importance = np.abs(W).flatten()

    return {
        'W': W.flatten(),
        'b': b,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'time': train_time,
        'feature_importance': feature_importance
    }

# Hyperparameters
LAMBDA_L2 = 0.01
ABSOLUTE_THRESHOLD = 0.01

# Train perceptron with L2
results_l2 = train_perceptron_l2(X_train, X_test, y_train, y_test, lambda_l2=LAMBDA_L2)
importance = results_l2['feature_importance']

# Feature selection (optional, similar to L1)
keep_features = importance > ABSOLUTE_THRESHOLD
print(f"\nOriginal features: {len(feature_names)}")
print(f"Retained features (> {ABSOLUTE_THRESHOLD}): {np.sum(keep_features)}")

for idx, imp in enumerate(importance):
    status = "RETAINED" if imp > ABSOLUTE_THRESHOLD else "ELIMINATED"
    print(f"{idx:2d}. {feature_names[idx]:<40} Importance={imp:.4f} => {status}")

# Reduce dataset
X_train_sparse = X_train[:, keep_features]
X_test_sparse = X_test[:, keep_features]

# Retrain on selected features without regularization
results_sparse = train_perceptron_l2(X_train_sparse, X_test_sparse, y_train, y_test, lambda_l2=0.0)

# Comparison
print("\n=== Performance Comparison ===")
print(f"{'Model':<20} {'Train Acc':<10} {'Test Acc':<10} {'Time(s)':<10} {'# Features'}")
print("-"*65)
print(f"L2 Reg          {results_l2['train_acc']:.4f}     {results_l2['test_acc']:.4f}     {results_l2['time']:.2f}      {len(feature_names)}")
print(f"L2 Sparse       {results_sparse['train_acc']:.4f}     {results_sparse['test_acc']:.4f}     {results_sparse['time']:.2f}      {X_train_sparse.shape[1]}")


Train=455, Test=114, Features=30

Original features: 30
Retained features (> 0.01): 30
 0. mean radius                              Importance=0.2713 => RETAINED
 1. mean texture                             Importance=0.2930 => RETAINED
 2. mean perimeter                           Importance=0.2626 => RETAINED
 3. mean area                                Importance=0.2557 => RETAINED
 4. mean smoothness                          Importance=0.1144 => RETAINED
 5. mean compactness                         Importance=0.0271 => RETAINED
 6. mean concavity                           Importance=0.1944 => RETAINED
 7. mean concave points                      Importance=0.2590 => RETAINED
 8. mean symmetry                            Importance=0.1113 => RETAINED
 9. mean fractal dimension                   Importance=0.1499 => RETAINED
10. radius error                             Importance=0.2494 => RETAINED
11. texture error                            Importance=0.0533 => RETAINED
12. perimeter